In [11]:
# Import Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
import time
import copy
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 1. Data Preparation

In [14]:
# Define data directories
data_dir = 'dataset'
train_dir = f'{data_dir}/train/train'
valid_dir = f'{data_dir}/valid'

# Data augmentation and normalization for training
# Just normalization for validation
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Custom function to filter out 'test' folder
def is_valid_folder(folder_name):
    return folder_name.lower() != 'test'

# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)

# For validation, we need to manually filter out the 'test' folder
import os
from torchvision.datasets.folder import default_loader, IMG_EXTENSIONS

# Get only valid class folders (exclude 'test')
valid_classes = [d for d in os.listdir(valid_dir) 
                if os.path.isdir(os.path.join(valid_dir, d)) and is_valid_folder(d)]
valid_classes.sort()

# Create a temporary mapping that excludes 'test'
class_to_idx = {cls_name: i for i, cls_name in enumerate(valid_classes)}

# Load validation dataset with filtered classes
val_dataset = datasets.ImageFolder(valid_dir, transform=val_transforms, 
                                    is_valid_file=lambda path: os.path.basename(os.path.dirname(path)).lower() != 'test')

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# Get class names
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"\nClass names: {class_names}")


FileNotFoundError: Found no valid file for the classes test. 

## 2. Training and Evaluation Functions

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=10):
    """
    Train the model and track best performance
    """
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    train_loss_history = []
    train_acc_history = []
    val_loss_history = []
    val_acc_history = []
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 50)
        
        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader
            
            running_loss = 0.0
            running_corrects = 0
            
            # Iterate over data
            for inputs, labels in tqdm(dataloader, desc=f'{phase.capitalize()}'):
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                # Zero the parameter gradients
                optimizer.zero_grad()
                
                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    # Backward + optimize only in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)
            
            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            # Store history
            if phase == 'train':
                train_loss_history.append(epoch_loss)
                train_acc_history.append(epoch_acc.item())
            else:
                val_loss_history.append(epoch_loss)
                val_acc_history.append(epoch_acc.item())
                
                # Deep copy the model if it's the best
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
    
    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Acc: {best_acc:.4f}')
    
    # Load best model weights
    model.load_state_dict(best_model_wts)
    
    history = {
        'train_loss': train_loss_history,
        'train_acc': train_acc_history,
        'val_loss': val_loss_history,
        'val_acc': val_acc_history
    }
    
    return model, history

In [ ]:
def evaluate_model(model, dataloader):
    """
    Evaluate model and return predictions and true labels
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc='Evaluating'):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)


def calculate_metrics(y_true, y_pred, model_name):
    """
    Calculate and print all evaluation metrics
    """
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"\n{'='*60}")
    print(f"{model_name} - Evaluation Metrics")
    print(f"{'='*60}")
    print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"{'='*60}\n")
    
    # Classification report
    print(f"\n{model_name} - Detailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }


def plot_confusion_matrix(y_true, y_pred, model_name):
    """
    Plot confusion matrix
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(16, 14))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.title(f'{model_name} - Confusion Matrix', fontsize=16, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    return cm

## 3. ResNet18 Model

In [ ]:
# Load pretrained ResNet18
print("Loading ResNet18 model...")
resnet_model = models.resnet18(pretrained=True)

# Freeze all layers
for param in resnet_model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, num_classes)
)

# Move model to device
resnet_model = resnet_model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
resnet_optimizer = optim.Adam(resnet_model.fc.parameters(), lr=0.001)

print(f"ResNet18 model loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in resnet_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in resnet_model.parameters() if p.requires_grad):,}")

### 3.1 Train ResNet18

In [ ]:
# Train ResNet18
print("Training ResNet18...")
num_epochs = 10  # Adjust as needed

resnet_model, resnet_history = train_model(
    resnet_model, 
    criterion, 
    resnet_optimizer, 
    num_epochs=num_epochs
)

# Save the trained model
torch.save(resnet_model.state_dict(), 'resnet18_fruit_classifier.pth')
print("ResNet18 model saved!")

### 3.2 Evaluate ResNet18

In [ ]:
# Evaluate ResNet18 on validation set
print("Evaluating ResNet18...")
resnet_preds, resnet_labels = evaluate_model(resnet_model, val_loader)

# Calculate metrics
resnet_metrics = calculate_metrics(resnet_labels, resnet_preds, "ResNet18")

# Plot confusion matrix
resnet_cm = plot_confusion_matrix(resnet_labels, resnet_preds, "ResNet18")

## 4. EfficientNet-B0 Model

In [ ]:
# Load pretrained EfficientNet-B0
print("Loading EfficientNet-B0 model...")
efficientnet_model = models.efficientnet_b0(pretrained=True)

# Freeze all layers
for param in efficientnet_model.parameters():
    param.requires_grad = False

# Replace the classifier
num_ftrs = efficientnet_model.classifier[1].in_features
efficientnet_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(num_ftrs, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, num_classes)
)

# Move model to device
efficientnet_model = efficientnet_model.to(device)

# Define optimizer
efficientnet_optimizer = optim.Adam(efficientnet_model.classifier.parameters(), lr=0.001)

print(f"EfficientNet-B0 model loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in efficientnet_model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in efficientnet_model.parameters() if p.requires_grad):,}")

### 4.1 Train EfficientNet-B0

In [ ]:
# Train EfficientNet-B0
print("Training EfficientNet-B0...")

efficientnet_model, efficientnet_history = train_model(
    efficientnet_model, 
    criterion, 
    efficientnet_optimizer, 
    num_epochs=num_epochs
)

# Save the trained model
torch.save(efficientnet_model.state_dict(), 'efficientnet_b0_fruit_classifier.pth')
print("EfficientNet-B0 model saved!")

### 4.2 Evaluate EfficientNet-B0

In [ ]:
# Evaluate EfficientNet-B0 on validation set
print("Evaluating EfficientNet-B0...")
efficientnet_preds, efficientnet_labels = evaluate_model(efficientnet_model, val_loader)

# Calculate metrics
efficientnet_metrics = calculate_metrics(efficientnet_labels, efficientnet_preds, "EfficientNet-B0")

# Plot confusion matrix
efficientnet_cm = plot_confusion_matrix(efficientnet_labels, efficientnet_preds, "EfficientNet-B0")

## 5. Model Comparison

In [ ]:
# Comparison table
import pandas as pd

comparison_df = pd.DataFrame({
    'Model': ['ResNet18', 'EfficientNet-B0'],
    'Accuracy': [resnet_metrics['accuracy'], efficientnet_metrics['accuracy']],
    'Precision': [resnet_metrics['precision'], efficientnet_metrics['precision']],
    'Recall': [resnet_metrics['recall'], efficientnet_metrics['recall']],
    'F1-Score': [resnet_metrics['f1_score'], efficientnet_metrics['f1_score']]
})

print("\n" + "="*80)
print("PERBANDINGAN PERFORMA MODEL")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Determine best model
best_model_idx = comparison_df['Accuracy'].idxmax()
best_model = comparison_df.loc[best_model_idx, 'Model']
best_accuracy = comparison_df.loc[best_model_idx, 'Accuracy']

print(f"\n🏆 Model terbaik: {best_model} dengan akurasi {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
print("="*80)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Metrics comparison
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics_names))
width = 0.35

resnet_values = [resnet_metrics['accuracy'], resnet_metrics['precision'], 
                 resnet_metrics['recall'], resnet_metrics['f1_score']]
efficientnet_values = [efficientnet_metrics['accuracy'], efficientnet_metrics['precision'], 
                       efficientnet_metrics['recall'], efficientnet_metrics['f1_score']]

axes[0].bar(x - width/2, resnet_values, width, label='ResNet18', color='#3498db', alpha=0.8)
axes[0].bar(x + width/2, efficientnet_values, width, label='EfficientNet-B0', color='#e74c3c', alpha=0.8)
axes[0].set_xlabel('Metrics', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[0].set_title('Performance Metrics Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0, 1.05])

# Add value labels on bars
for i, (rv, ev) in enumerate(zip(resnet_values, efficientnet_values)):
    axes[0].text(i - width/2, rv + 0.01, f'{rv:.3f}', ha='center', va='bottom', fontsize=9)
    axes[0].text(i + width/2, ev + 0.01, f'{ev:.3f}', ha='center', va='bottom', fontsize=9)

# Training history comparison (Accuracy)
epochs_range = range(1, len(resnet_history['train_acc']) + 1)
axes[1].plot(epochs_range, resnet_history['val_acc'], 'b-', label='ResNet18 Val', linewidth=2)
axes[1].plot(epochs_range, efficientnet_history['val_acc'], 'r-', label='EfficientNet-B0 Val', linewidth=2)
axes[1].plot(epochs_range, resnet_history['train_acc'], 'b--', label='ResNet18 Train', linewidth=1, alpha=0.5)
axes[1].plot(epochs_range, efficientnet_history['train_acc'], 'r--', label='EfficientNet-B0 Train', linewidth=1, alpha=0.5)
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[1].set_title('Training History Comparison', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Kesimpulan Analisis

### Perbedaan Arsitektur:

**ResNet18:**
- Menggunakan skip connections (residual connections) untuk mengatasi vanishing gradient
- Arsitektur lebih sederhana dengan 18 layer
- Parameter lebih sedikit, training lebih cepat
- Cocok untuk dataset medium-sized

**EfficientNet-B0:**
- Menggunakan compound scaling (depth, width, resolution)
- Arsitektur lebih kompleks dan efisien
- Lebih banyak parameter tetapi dioptimalkan untuk efisiensi
- Menggunakan mobile inverted bottleneck convolution (MBConv)
- Umumnya memberikan performa lebih baik dengan parameter yang lebih sedikit

### Analisis Performa:
Berdasarkan metrik evaluasi yang dihitung (Accuracy, Precision, Recall, F1-Score), dapat dilihat perbandingan performa antara kedua model untuk klasifikasi buah-buahan.